# CodeSearch-FT Phase 5 evaluation

This notebook runs the existing fixed-pool evaluation package. It does **not** train or modify the model.

Run cells in order. Select a Colab GPU, mount Drive, upload the evaluation bundle, then run the real-data smoke and bounded 128-item comparison. Evaluation outputs and the fixed-set manifest are written to Drive. Labels are automatic CodeSearchNet associations, not human judgments.

## 1. Verify Colab GPU and CUDA

In Colab, choose **Runtime → Change runtime type → T4 GPU** before running this cell.

In [ ]:
import platform
import sys

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then rerun this cell.")
DEVICE = "cuda:0"
print("GPU:", torch.cuda.get_device_name(0))
assert (3, 11) <= sys.version_info[:2] < (3, 14), "The wheel supports Python 3.11–3.13."

## 2. Mount Google Drive for persistent results and the existing checkpoint

This is a user-run Colab action. The notebook only accesses Drive after you run this cell.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/codesearch_ft_colab_pilot")
CHECKPOINT_EXPORT = DRIVE_ROOT / "pilot" / "export"
CHECKPOINT_MODEL = CHECKPOINT_EXPORT / "model"
EVALUATION_ROOT = DRIVE_ROOT / "evaluation"
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
assert (CHECKPOINT_EXPORT / "metadata.json").is_file(), (
    f"Missing checkpoint metadata: {CHECKPOINT_EXPORT}"
)
assert (CHECKPOINT_MODEL / "model.safetensors").is_file(), (
    f"Missing model weights: {CHECKPOINT_MODEL}"
)
print("Checkpoint export:", CHECKPOINT_EXPORT)
print("Results root:", EVALUATION_ROOT)

## 3. Upload the deployment ZIP from your computer

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one codesearch_ft_colab_evaluation_bundle.zip file.")
BUNDLE_PATH = Path("/content") / zip_names[0]
print("Uploaded:", BUNDLE_PATH)

## 4. Validate bundle hashes, wheel contents, and Python compatibility

In [ ]:
import hashlib
import json
import zipfile
from pathlib import PurePosixPath

from packaging.specifiers import SpecifierSet

BUNDLE_DIR = Path("/content/codesearch_ft_colab_evaluation_bundle")
EXPECTED_NAMES = {"evaluation.yaml", "requirements-colab-evaluation.txt"}


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


with zipfile.ZipFile(BUNDLE_PATH) as archive:
    bad_member = archive.testzip()
    assert bad_member is None, f"Corrupt ZIP member: {bad_member}"
    names = set(archive.namelist())
    assert len(names) == len(archive.namelist()), "Duplicate ZIP member names."
    assert all(PurePosixPath(name).name == name and name not in {".", ".."} for name in names), (
        "Unsafe ZIP path."
    )
    manifest = json.loads(archive.read("deployment-manifest.json"))
    assert names - {"deployment-manifest.json"} == set(manifest["files"]), (
        "Unexpected bundle contents."
    )
    assert names - {"deployment-manifest.json"} == EXPECTED_NAMES | {manifest["wheel_filename"]}
    assert SpecifierSet(manifest["requires_python"]).contains(platform.python_version())
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    archive.extractall(BUNDLE_DIR)
for name, entry in manifest["files"].items():
    assert sha256_file(BUNDLE_DIR / name) == entry["sha256"], f"SHA-256 mismatch: {name}"
WHEEL_PATH = BUNDLE_DIR / manifest["wheel_filename"]
with zipfile.ZipFile(WHEEL_PATH) as wheel:
    wheel_names = set(wheel.namelist())
    required = {"codesearch_ft/evaluation/benchmark.py", "codesearch_ft/evaluation/fixed_corpus.py"}
    assert required <= wheel_names, "Wheel is stale: Phase 5 evaluation modules are absent."
    assert b"encoder_device" in wheel.read("codesearch_ft/evaluation/benchmark.py")
print("Bundle SHA-256:", sha256_file(BUNDLE_PATH))
print("Wheel contains Phase 5 engine modules.")
print("Python compatibility:", manifest["requires_python"])

## 5. Install evaluation dependencies without replacing Colab CUDA PyTorch

In [ ]:
import subprocess
from importlib.metadata import version as distribution_version

torch_version_before = torch.__version__
requirements = (BUNDLE_DIR / "requirements-colab-evaluation.txt").read_text()
assert not any(line.strip().lower().startswith("torch") for line in requirements.splitlines()), (
    "Requirements must not install PyTorch."
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(BUNDLE_DIR / "requirements-colab-evaluation.txt"),
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-deps",
        str(WHEEL_PATH),
    ]
)
torch_version_after = distribution_version("torch")
assert torch_version_after == torch_version_before, (
    "PyTorch version changed during dependency installation."
)
assert torch.cuda.is_available(), "CUDA became unavailable after dependency installation."
print("Preserved CUDA PyTorch:", torch_version_after)
print("Python:", platform.python_version(), "| PyTorch:", torch_version_after)

## 6. Preflight imports, real FAISS CPU index, package config, and checkpoint

In [ ]:
import datasets
import faiss
import numpy as np
import transformers
import yaml

from codesearch_ft.config.models import EvaluationConfig
from codesearch_ft.datasets.codesearchnet import DATASET_ID, DATASET_REVISION
from codesearch_ft.evaluation.benchmark import run_benchmark
from codesearch_ft.evaluation.fixed_corpus import (
    EvaluationCandidate,
    EvaluationQuery,
    load_jsonl,
    prepare_fixed_evaluation_set,
)
from codesearch_ft.ml.codebert_encoder import MODEL_ID, MODEL_REVISION, CodeBERTEncoder

index = faiss.IndexFlatIP(2)
index.add(np.asarray([[1.0, 0.0]], dtype="float32"))
scores, ids = index.search(np.asarray([[1.0, 0.0]], dtype="float32"), 1)
assert int(ids[0, 0]) == 0 and float(scores[0, 0]) > 0.99
bundle_config = yaml.safe_load((BUNDLE_DIR / "evaluation.yaml").read_text())
assert (
    bundle_config["dataset_id"] == DATASET_ID
    and bundle_config["dataset_revision"] == DATASET_REVISION
)
assert bundle_config["model_revision"] == MODEL_REVISION
metadata = json.loads((CHECKPOINT_EXPORT / "metadata.json").read_text())
assert (
    metadata["dataset_revision"] == DATASET_REVISION
    and metadata["model_revision"] == MODEL_REVISION
)
print(
    "FAISS:",
    faiss.__version__,
    "| datasets:",
    datasets.__version__,
    "| transformers:",
    transformers.__version__,
)
print("Pinned dataset/model revisions and checkpoint metadata agree.")

## 7. Load pretrained and fine-tuned CodeBERT on the selected GPU

The pretrained model is fetched from its pinned public revision if it is not cached. The fine-tuned checkpoint is loaded from Drive. This cell loads weights only; it performs no training.

In [ ]:
checkpoint_hash = sha256_file(CHECKPOINT_MODEL / "model.safetensors")
print("Fine-tuned weights SHA-256:", checkpoint_hash)
pretrained_encoder = CodeBERTEncoder.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, device=DEVICE, local_files_only=False, batch_size=8
)
fine_tuned_encoder = CodeBERTEncoder.from_pretrained(
    CHECKPOINT_EXPORT.as_posix(),
    revision=MODEL_REVISION,
    device=DEVICE,
    local_files_only=True,
    batch_size=8,
)
probe_query = fine_tuned_encoder.encode_queries(
    ("remove duplicate values while preserving input order",)
)
probe_code = fine_tuned_encoder.encode_code(
    ("def unique(items):\n    return list(dict.fromkeys(items))",)
)
assert len(probe_query) == len(probe_code) == 1 and len(probe_query[0]) == 768
assert abs(sum(value * value for value in probe_query[0]) - 1.0) < 1e-4
print(
    "Checkpoint query/code embeddings verified:",
    len(probe_query[0]),
    "dimensions on",
    fine_tuned_encoder.device,
)

## 8. Load pinned held-out data and verify isolation and deterministic selection

The existing adapter reads bounded rows from all three official splits, removes cross-split exact duplicate groups, preserves split assignments, strips target docstrings, and creates traceable test-only query/code associations. The fixed set and detailed partition manifest are saved in Drive.

In [ ]:
import shutil
from dataclasses import asdict
from pathlib import Path

FULL_DIR = EVALUATION_ROOT / "full_run"
FIXED_DIR = FULL_DIR / "fixed"
OUTPUT_DIR = FULL_DIR / "outputs"
FULL_DIR.mkdir(parents=True, exist_ok=True)
config_data = dict(bundle_config)
config_data.update(
    candidate_corpus=str(FIXED_DIR / "candidates.jsonl"),
    queries=str(FIXED_DIR / "queries.jsonl"),
    output_dir=str(OUTPUT_DIR),
    fine_tuned_checkpoint=str(CHECKPOINT_EXPORT),
    candidate_limit=128,
    query_limit=128,
    random_seed=42,
)
full_config = EvaluationConfig.model_validate(config_data)
fixed_manifest = prepare_fixed_evaluation_set(full_config)
partition = fixed_manifest["partition_manifest"]
assert (
    fixed_manifest["official_split"] == "test"
    and fixed_manifest["dataset_revision"] == DATASET_REVISION
)
assert all(partition["row_counts"][split] > 0 for split in ("train", "validation", "test"))
assert partition["cross_split_duplicate_groups"] >= 0 and "Quarantine" in partition["policy"]
assert (
    fixed_manifest["counts"]["source_train_rows"] > 0
    and fixed_manifest["counts"]["source_validation_rows"] > 0
    and fixed_manifest["counts"]["source_test_rows"] > 0
)
assert sha256_file(full_config.candidate_corpus) == fixed_manifest["candidates_sha256"]
assert sha256_file(full_config.queries) == fixed_manifest["queries_sha256"]
repeat_dir = EVALUATION_ROOT / "_selection_repeat_check"
repeat_config = full_config.model_copy(
    update={
        "candidate_corpus": repeat_dir / "candidates.jsonl",
        "queries": repeat_dir / "queries.jsonl",
    }
)
repeat_manifest = prepare_fixed_evaluation_set(repeat_config)
assert repeat_manifest["fixed_set_sha256"] == fixed_manifest["fixed_set_sha256"]
assert repeat_config.candidate_corpus.read_bytes() == full_config.candidate_corpus.read_bytes()
assert repeat_config.queries.read_bytes() == full_config.queries.read_bytes()
shutil.rmtree(repeat_dir)
print(
    json.dumps(
        {
            "source_rows": partition["row_counts"],
            "cross_split_duplicate_groups": partition["cross_split_duplicate_groups"],
            "quarantined_rows": partition["removed_counts"],
            "candidate_count": fixed_manifest["counts"]["unique_candidates"],
            "query_count": fixed_manifest["counts"]["unique_queries"],
            "fixed_set_sha256": fixed_manifest["fixed_set_sha256"],
        },
        indent=2,
    )
)

## 9. Run a bounded real-data smoke benchmark

This smoke uses the first eight fixed test queries and at most sixteen candidates, retaining each query's known positive candidate. All four modes run through the same existing benchmark engine and same pool.

In [ ]:
smoke_dir = EVALUATION_ROOT / "smoke_run"
smoke_fixed_dir = smoke_dir / "fixed"
smoke_output_dir = smoke_dir / "outputs"
smoke_fixed_dir.mkdir(parents=True, exist_ok=True)
smoke_output_dir.mkdir(parents=True, exist_ok=True)
all_candidates = load_jsonl(full_config.candidate_corpus, EvaluationCandidate)
all_queries = load_jsonl(full_config.queries, EvaluationQuery)
smoke_queries = all_queries[:8]
positive_ids = {candidate_id for query in smoke_queries for candidate_id in query.relevance}
assert 0 < len(positive_ids) <= 16
positive_candidates = [item for item in all_candidates if item.candidate_id in positive_ids]
fill_candidates = [item for item in all_candidates if item.candidate_id not in positive_ids]
smoke_candidates = positive_candidates + fill_candidates[: max(0, 16 - len(positive_candidates))]
smoke_candidate_ids = {item.candidate_id for item in smoke_candidates}
smoke_candidate_bytes = "".join(
    json.dumps(asdict(item), sort_keys=True) + "\n" for item in smoke_candidates
).encode()
smoke_query_bytes = "".join(
    json.dumps(asdict(item), sort_keys=True) + "\n" for item in smoke_queries
).encode()
(smoke_fixed_dir / "candidates.jsonl").write_bytes(smoke_candidate_bytes)
(smoke_fixed_dir / "queries.jsonl").write_bytes(smoke_query_bytes)
smoke_manifest = dict(fixed_manifest)
smoke_manifest["counts"] = dict(fixed_manifest["counts"])
smoke_manifest["counts"].update(
    unique_candidates=len(smoke_candidates),
    unique_queries=len(smoke_queries),
    queries_with_traceable_positive=len(smoke_queries),
)
smoke_selection = (
    "First eight fixed queries, all their automatic positive candidates, "
    "then source-order fill to sixteen candidates."
)
smoke_manifest.update(
    candidate_limit=len(smoke_candidates),
    query_limit=len(smoke_queries),
    candidates=[item.candidate_id for item in smoke_candidates],
    queries=[item.query_id for item in smoke_queries],
    selection=smoke_selection,
    candidates_sha256=hashlib.sha256(smoke_candidate_bytes).hexdigest(),
    queries_sha256=hashlib.sha256(smoke_query_bytes).hexdigest(),
    parent_fixed_set_sha256=fixed_manifest["fixed_set_sha256"],
)
smoke_manifest.pop("fixed_set_sha256", None)
smoke_manifest["fixed_set_sha256"] = hashlib.sha256(
    json.dumps(smoke_manifest, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
(smoke_fixed_dir / "dataset-manifest.json").write_text(
    json.dumps(smoke_manifest, indent=2, sort_keys=True) + "\n"
)
smoke_config = full_config.model_copy(
    update={
        "candidate_corpus": smoke_fixed_dir / "candidates.jsonl",
        "queries": smoke_fixed_dir / "queries.jsonl",
        "output_dir": smoke_output_dir,
        "candidate_limit": len(smoke_candidates),
        "query_limit": len(smoke_queries),
    }
)
smoke_report = run_benchmark(
    smoke_config, pretrained_encoder=pretrained_encoder, fine_tuned_encoder=fine_tuned_encoder
)
assert len(smoke_report["dataset"]["candidate_id_set_sha256"]) == 64
assert len({value["candidate_pool_sha256"] for value in smoke_report["results"].values()}) == 1
assert set(smoke_report["results"]) == {
    "bm25",
    "pretrained_codebert",
    "fine_tuned_codebert",
    "hybrid",
}
print(
    "Real CodeSearchNet smoke complete:",
    smoke_report["dataset"]["candidate_count"],
    "candidates,",
    smoke_report["dataset"]["query_count"],
    "queries.",
)

## 10. Run the bounded Phase 5 comparison

This is the actual benchmark, capped by the bundled configuration at 128 candidates and 128 queries. It does not train or tune any model. Allow the cell to finish; its outputs include measured CPU FAISS/SQLite indexing and query latencies plus GPU encoder use.

In [ ]:
import contextlib
import sys


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return False


FULL_DIR.mkdir(parents=True, exist_ok=True)
(FULL_DIR / "evaluation-config.yaml").write_text(
    yaml.safe_dump(full_config.model_dump(mode="json"), sort_keys=True)
)
(FULL_DIR / "deployment-manifest.json").write_bytes(
    (BUNDLE_DIR / "deployment-manifest.json").read_bytes()
)
runtime_info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cuda_device": torch.cuda.get_device_name(0),
    "encoder_device": str(fine_tuned_encoder.device),
    "faiss": faiss.__version__,
    "datasets": datasets.__version__,
    "transformers": transformers.__version__,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "checkpoint_weights_sha256": checkpoint_hash,
}
(FULL_DIR / "runtime-environment.json").write_text(
    json.dumps(runtime_info, indent=2, sort_keys=True) + "\n"
)
with (
    (FULL_DIR / "benchmark.log").open("w", encoding="utf-8") as log_file,
    contextlib.redirect_stdout(Tee(sys.stdout, log_file)),
    contextlib.redirect_stderr(Tee(sys.stderr, log_file)),
):
    report = run_benchmark(
        full_config, pretrained_encoder=pretrained_encoder, fine_tuned_encoder=fine_tuned_encoder
    )
assert report["dataset"]["candidate_count"] <= 128 and report["dataset"]["query_count"] <= 128
pool_hashes = {value["candidate_pool_sha256"] for value in report["results"].values()}
assert len(pool_hashes) == 1
print("Encoder device recorded by engine:", report["environment"]["encoder_device"])
print("Saved benchmark outputs to:", OUTPUT_DIR)

## 11. Inspect metrics and report bootstrap uncertainty estimates

The bootstrap intervals resample evaluation queries and are descriptive only: source-order selection, automatic labels, and possible query dependence limit population-level interpretation. Latency is hardware/runtime specific.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

rng = np.random.default_rng(4205)
bootstrap_replicates = 5000
rows = []
for mode, summary in report["results"].items():
    query_rows = [row["modes"][mode] for row in report["per_query"]]
    mrr = np.asarray([row["reciprocal_rank_at_10"] for row in query_rows], dtype=float)
    recall = np.asarray([row["recall_at_10"] for row in query_rows], dtype=float)
    latency_ms = np.asarray([row["latency_seconds"] * 1000 for row in query_rows], dtype=float)
    indices = rng.integers(0, len(query_rows), size=(bootstrap_replicates, len(query_rows)))
    mrr_ci = np.quantile(mrr[indices].mean(axis=1), [0.025, 0.975])
    recall_ci = np.quantile(recall[indices].mean(axis=1), [0.025, 0.975])
    latency_ci = np.quantile(latency_ms[indices].mean(axis=1), [0.025, 0.975])
    rows.append(
        {
            "mode": mode,
            "MRR@10": summary["mrr_at_10"],
            "MRR@10 95% bootstrap CI": [float(x) for x in mrr_ci],
            "Recall@10": summary["recall_at_10"],
            "Recall@10 95% bootstrap CI": [float(x) for x in recall_ci],
            "mean latency ms": float(latency_ms.mean()),
            "mean latency 95% bootstrap CI ms": [float(x) for x in latency_ci],
            "measured p95 latency ms": summary["latency_p95_seconds"] * 1000,
            "queries": summary["query_count"],
        }
    )
uncertainty_interpretation = (
    "Descriptive resampling uncertainty only; fixed source-order subset and "
    "automatic associations limit population-level claims."
)
uncertainty = {
    "method": "Percentile bootstrap over query rows; 5000 replicates; seed 4205.",
    "interpretation": uncertainty_interpretation,
    "results": rows,
}
(OUTPUT_DIR / "uncertainty.json").write_text(
    json.dumps(uncertainty, indent=2, sort_keys=True) + "\n"
)
metrics_table = pd.DataFrame(rows)
display(metrics_table)
print("Markdown report:", OUTPUT_DIR / "benchmark.md")
print(
    "JSON/CSV/SVG:",
    OUTPUT_DIR / "benchmark.json",
    OUTPUT_DIR / "benchmark.csv",
    OUTPUT_DIR / "metrics.svg",
)
print("Fixed set and partition manifest:", FIXED_DIR / "dataset-manifest.json")
print("Saved query-level results, indexes, logs, configuration, runtime, and uncertainty to Drive.")

## 12. Recover results and checkpoint

All persisted evaluation files are under the Drive paths printed above. The fine-tuned checkpoint remains in its existing pilot export directory. To recover it from Drive in a later Colab session, mount Drive again and point CodeSearch-FT at that same export directory; no training is needed.

In [ ]:
print("Evaluation results:", EVALUATION_ROOT)
print("Fine-tuned model:", CHECKPOINT_MODEL)
print("Model weights:", CHECKPOINT_MODEL / "model.safetensors")
print(
    "Tokenizer files:", sorted(path.name for path in CHECKPOINT_MODEL.iterdir() if path.is_file())
)
print("Benchmark report:", OUTPUT_DIR / "benchmark.md")
print("Raw per-query results:", OUTPUT_DIR / "benchmark.json")